# Agent nodes

![Multi-agent flow](multi_agent_flow.png)

An **Agent** node calls a Bedrock Agent from inside a flow. `multi_agent_flow.json` chains two agents: the first agent's answer becomes the second agent's input. This notebook creates both agents for you:

- **Researcher** has one tool, `search_aws_docs`, which searches AWS documentation through the public AWS Knowledge MCP Server. It is the same Lambda function as module 03. The agent decides whether to search, what to search for, and whether to search again.
- **Editor** has no tools. It rewrites the Researcher's answer as three bullet points plus sources.

> **Teaching/Learning Tip:** compare this with module 03. There, the *flow* decides the route with a Condition node, and every AWS question gets exactly one search. Here, the *agent* decides: it skips the tool for a non-AWS question, and for an AWS question it may search, judge the results, and search again with a narrower query. That decision loop is what makes it an agent.

> **Status (November 2026):** Amazon Bedrock Agents is now **Agents Classic**, and it is in [maintenance mode](https://docs.aws.amazon.com/bedrock/latest/userguide/agents-classic-maintenance-mode.html):
> - It is closed to new customers. Accounts that already use it can still create agents. If `create_agent` fails for your account, skip this module.
> - It gets no new features, and that includes support for newer models. This notebook shows it. Nova 2 Lite doesn't follow the default orchestration prompt reliably, and Claude Haiku 5.5 rejects the thinking settings Agents Classic sends. So the agents use **Claude Haiku 4.5**. Haiku 4.5's model card lists its end of life as "no sooner than Oct 16, 2026". It is still active, but when AWS sets a retirement date, this module will need another model that Agents Classic accepts.
> - AWS recommends **Amazon Bedrock AgentCore** for new agent projects. There is no deadline to migrate.
>
> The flow concepts here (agent nodes, aliases, tools, tracing) still apply. Treat the agent setup itself as legacy.

> **Multi-turn:** `multi_turn_agent_flow.json` (also in this folder) holds a back-and-forth conversation with one agent. It needs interactive input, so run it with the flow manager in `tools/flow_manager`.

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.append("..")  # flow_utils.py lives one folder up
import flow_utils

session, region, account = flow_utils.get_session_info()  # from your current AWS login
agent = session.client("bedrock-agent")            # control plane: flows and agents
runtime = session.client("bedrock-agent-runtime")  # data plane: invoke flows and agents
lambda_client = session.client("lambda")
iam = session.client("iam")

role_arn = flow_utils.ensure_role(iam, region, account)              # the flow's role: may invoke agent aliases
agent_role_arn = flow_utils.ensure_agent_role(iam, region, account)  # the agents' role: may invoke the model

## Choose the agents' model

The agents use **Claude Haiku 4.5**, not the Nova 2 Lite model the other modules use. See the status note at the top for why the choice is limited.

> **Teaching/Learning Tip:** an agent runs an orchestration prompt that asks the model to either call a tool or give a final answer in a set format. With Agents Classic's default prompt, Nova 2 Lite sometimes ends a turn with neither when it wants to search again. The trace shows `LLM prediction format incorrect`, and the agent replies "Sorry, I am unable to assist you with this request." Claude Haiku 4.5 follows the format reliably. Newer Claude models, such as Haiku 5.5, reject the thinking settings that Agents Classic sends.

In [ ]:
model_id = flow_utils.default_agent_model_id(region, account)  # Claude Haiku 4.5 inference profile for this region
print("agent model:", model_id)

## Define the agents and the tool

The tool is an **action group** defined with *function details*: a name, a description, and typed parameters. The model reads the descriptions to decide when to call it and what to pass.

In [ ]:
LAMBDA_NAME = "BedrockFlowsSampleAgentDocsTool"
LAMBDA_SOURCE = Path("..") / "03_conditions" / "aws_docs_lambda.py"  # shared with module 03

RESEARCHER = ("BedrockFlowsSampleResearcher",
              "You are an AWS expert researcher. For questions about AWS, use the search_aws_docs tool to find "
              "relevant AWS documentation before you answer. If the results don't answer the question, search "
              "again with a narrower query, but search at most three times, then answer with what you found. For questions that are not about AWS, answer from your own knowledge "
              "without the tool. Answer in one short paragraph of plain text, then list the source URLs you used.")
EDITOR = ("BedrockFlowsSampleEditor",
          "You are an editor with no tools. The user's message is a draft answer. Rewrite it as exactly three "
          "concise bullet points that keep its key facts. If the draft lists source URLs, keep them in a "
          "'Sources:' list after the bullets. Return only the bullets and sources.")
DOCS_TOOL = {
    "actionGroupName": "aws_docs",
    "description": "Search official AWS documentation.",
    "functionSchema": {"functions": [{
        "name": "search_aws_docs",
        "description": "Search AWS documentation, blog posts, and What's New posts. Returns excerpts with source URLs.",
        "parameters": {"query": {
            "type": "string",
            "description": "Search phrase, for example 'Bedrock flow Lambda node permissions'.",
            "required": True,
        }},
    }]},
}

## Deploy the tool and create the agents

> **Teaching/Learning Tip:** an agent calls its Lambda function through a **resource-based policy** on the function, scoped to that one agent. `allow_agent_to_invoke_lambda` adds it after the agent exists and before it is prepared. A flow's Lambda node (module 03) works the other way: the flow's execution role gets `lambda:InvokeFunction`.

Creating and preparing two agents takes a minute or two.

In [ ]:
lambda_arn = flow_utils.deploy_lambda(lambda_client, iam, LAMBDA_NAME, LAMBDA_SOURCE)
tool = dict(DOCS_TOOL, actionGroupExecutor={"lambda": lambda_arn})

created = []  # (agent_id, alias_id) for cleanup
researcher_id, researcher_alias_id, researcher_arn = flow_utils.create_agent(
    agent, *RESEARCHER, model_id, agent_role_arn, action_groups=[tool],
    before_prepare=lambda agent_id: flow_utils.allow_agent_to_invoke_lambda(
        lambda_client, LAMBDA_NAME, region, account, agent_id))
created.append((researcher_id, researcher_alias_id))
editor_id, editor_alias_id, editor_arn = flow_utils.create_agent(agent, *EDITOR, model_id, agent_role_arn)
created.append((editor_id, editor_alias_id))

## Watch the Researcher decide

Invoke the Researcher directly with tracing on. `invoke_agent` prints its reasoning (`[thinks]`), each tool call (`[calls]`), and what came back (`[gets]`). Expect one or more searches for the AWS question and none for the goldfish question.

In [ ]:
for question in ["What does a Bedrock flow's execution role need to call a Lambda function?",
                 "What is a good name for a pet goldfish?"]:
    print(f"\nResearcher, directly: {question}")
    answer = flow_utils.invoke_agent(runtime, researcher_id, researcher_alias_id, question)
    print(f"  [answer] {answer}")

## Put both agents in a flow

The two placeholders are the agents' alias ARNs. The question goes to the Researcher, its answer goes to the Editor, and the flow returns the Editor's bullets.

> **Teaching/Learning Tip:** a flow calls an agent *alias*, not the agent itself, so you can move the alias to a new agent version without changing the flow. The console's built-in test alias (`TSTALIASID`) does not work in a flow. To see each agent's tool calls in the console, open the flow, run a test, and look at the trace.

In [ ]:
template = flow_utils.load_template("multi_agent_flow.json", {
    "$$AGENT_1_ALIAS_ARN": researcher_arn,
    "$$AGENT_2_ALIAS_ARN": editor_arn,
})
flow_id = flow_utils.create_flow(
    agent, template["name"], template["description"], template["definition"], role_arn, template.get("tags"))
version, alias_id = flow_utils.deploy_flow(agent, flow_id)

question = "How do I invoke a Bedrock flow from my application?"
outputs = flow_utils.invoke_flow(runtime, flow_id, alias_id, question)
print(f"Flow (Researcher -> Editor): {question}\n{outputs[0]}")

## Cleanup

Delete the flow (alias, version, flow), each agent (alias, then agent), and the Lambda function.
The sample IAM roles are left in place so other modules can reuse them. Delete them with
`flow_utils.delete_role(iam)`, `flow_utils.delete_agent_role(iam)`, and `flow_utils.delete_lambda_role(iam)` when you are completely done.

In [ ]:
flow_utils.cleanup_flow(agent, flow_id, alias_id, version)
for agent_id, agent_alias_id in created:
    flow_utils.delete_agent(agent, agent_id, agent_alias_id)
flow_utils.delete_lambda(lambda_client, LAMBDA_NAME)